# 🌌 NeoWatch — Aşama 2: Keşifsel Veri Analizi (EDA) ve Özellik Mühendisliği

**Plan Belgesi**: NASA Proje Planı 2 (Aşama 2)

### 🎯 Hedefler:
1. **Eksik ve Aykırı Değer (Outlier) Analizi**: Kutu Grafiği (Boxplot) ile görselleştirme ve fiziksel sınırları koruyarak IQR yöntemiyle aykırı değer traşlama (capping).
2. **Korelasyon ve Çoklu Doğrusallık (Multicollinearity)**: `seaborn.heatmap` ile değişkenler arası ilişkileri inceleme, yüksek korelasyonlu minimum ve maksimum çapı birleştirerek `estimated_diameter_mean_km` özniteliğini üretme.
3. **Birim Ölçeklendirme (Scaling)**: Mesafe ve hız birim farklılıklarını gidermek için `StandardScaler` nesnesini veri sızıntısı (data leakage) olmadan sadece eğitim verisinde eğitme (`fit_transform`).
4. **Dengesiz Veri (Imbalanced Data) Çözümü**: Azınlıkta olan (%11) tehlikeli asteroit sınıfını `imblearn.over_sampling.SMOTE` ile sentetik örnekler üreterek dengeleme.
5. **Checkpoint 2 (Modele Hazır Veri)**: Temizlenmiş ve ölçeklenmiş veriyi `data/processed_data.csv`, dönüştürücü algoritmayı ise `models/scaler.pkl` olarak kaydetme.

In [ ]:
import sys
import os
from pathlib import Path

project_root = Path(os.path.abspath('')).parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from src.config import (
    RAW_DATA_PATH,
    PROCESSED_DATA_PATH,
    LEGACY_RAW_DATA_PATH,
    SCALER_PATH,
    FEATURE_COLUMNS,
    TARGET_COLUMN,
)
from src.data_processor import DataProcessor

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("Aşama 2 Çalışma Ortamı Hazırlandı!")

## 1. Ham Veriyi Yükleme ve Hedef Sınıf Dağılımı

In [ ]:
raw_path = RAW_DATA_PATH if RAW_DATA_PATH.exists() else LEGACY_RAW_DATA_PATH
df_raw = pd.read_csv(raw_path)
print(f"Yüklenen ham veri boyutu: {df_raw.shape}")

# Hedef Değişken Dağılım Grafiği
plt.figure(figsize=(6, 4))
sns.countplot(data=df_raw, x=TARGET_COLUMN, palette=['#2563eb', '#dc2626'])
plt.title('Hedef Değişken Dağılımı (0: Güvenli, 1: Tehlikeli Asteroit)')
plt.xticks([0, 1], ['Güvenli (Safe)', 'Potansiyel Tehlikeli (PHA)'])
plt.show()

print("Sınıf Oranları:")
print(df_raw[TARGET_COLUMN].value_counts(normalize=True) * 100)

## 2. Eksik Değer ve Aykırı Değer (Outlier) Analizi (Boxplot & IQR)

In [ ]:
processor = DataProcessor(scaler_type='standard')
df_clean = processor.clean_raw_data(df_raw)

# Kutu Grafiği (Boxplot) ile Sürekli Değişkenlerin Dağılımı
plot_cols = [c for c in FEATURE_COLUMNS if c in df_clean.columns]
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(plot_cols):
    sns.boxplot(data=df_clean, y=col, x=TARGET_COLUMN, ax=axes[i], palette=['#3b82f6', '#ef4444'])
    axes[i].set_title(f'{col} Dağılımı')

plt.tight_layout()
plt.show()

# IQR Sınır Değerleri
iqr_bounds = processor.calculate_iqr_bounds(df_clean, factor=3.0)
print("--- IQR Sınır Değerleri ---")
for feat, (lb, ub) in iqr_bounds.items():
    print(f"{feat:30s} -> Alt: {lb:.4f}, Üst: {ub:.4f}")

## 3. Korelasyon ve Çoklu Doğrusallık (Multicollinearity) Analizi

`tahmini_min_çap` ve `tahmini_maks_çap` birbirleriyle %99 oranında korelasyona sahiptir. Model stabilitesini sağlamak için ikisinin ortalaması olan `estimated_diameter_mean_km` değişkeni üretilmektedir.

In [ ]:
plt.figure(figsize=(10, 7))
corr_matrix = df_clean[FEATURE_COLUMNS + [TARGET_COLUMN]].corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Öznitelik Korelasyon Matrisi (Heatmap)')
plt.show()

## 4. StandardScaler Ölçeklendirme ve SMOTE ile Sınıf Dengeleme (CHECKPOINT 2)

In [ ]:
# Uçtan uca Aşama 2 boru hattını çalıştırma
X_train_resampled, y_train_resampled, X_test_scaled, y_test, df_processed = processor.prepare_datasets(
    raw_csv_path=raw_path,
    test_size=0.2,
    random_state=42,
    apply_smote=True,
)

print(f"SMOTE Sonrası Eğitim Kümesi Boyutu : {X_train_resampled.shape} (Tehlikeli: {sum(y_train_resampled==1)} / Güvenli: {sum(y_train_resampled==0)})")
print(f"Ölçeklenmiş Test Kümesi Boyutu    : {X_test_scaled.shape} (Tehlikeli: {sum(y_test==1)} / Güvenli: {sum(y_test==0)})")
print(f"\n🎯 CHECKPOINT 2 TAMAMLANDI:")
print(f" - İşlenmiş Veri Seti: {PROCESSED_DATA_PATH}")
print(f" - Scaler Nesnesi    : {SCALER_PATH}")